# Plots

In [ ]:
import json, os, csv
from datetime import datetime

import matplotlib.pyplot as plt


def list_intersection(list1, list2):
    return list(set(list1) & set(list2))


def list_difference(list1, list2):
    return list(set(list1) - set(list2))


file_path = "accepted_entities.json"
with open(file_path, "r", encoding="utf-8") as file:
    entities = json.load(file)
high_freq = list(entities["GPE"]["high"].keys()) + list(
    entities["PERSON"]["high"].keys()
)
low_freq = list(entities["GPE"]["low"].keys()) + list(entities["PERSON"]["low"].keys())

## Main Stats

In [ ]:
experiments = {
    "VFT": "Vanilla FineTuning (LoRA)",
    "VFT_ENAF": "+Structured Annotation",
    "EFT": "+Adaptive Exposure",
    "VFT_KL_10": "+KL Mj|M0",
    "VFT_KL_LAST_1": "+KL MJ|Mj-1",
    "rag_all": "RAG",
    "rag_entity": "RAG Main Documents",
    "rag_gold": "RAG Gold Documents",
    #"VFT_MAIN": "Vanilla FineTuning (LoRA) on Main Docs",

}
epochs = [
    "epoch_1",
    "epoch_2",
    "epoch_3",
    "epoch_4",
    "epoch_5",
    "epoch_6",
    "epoch_7",
    "epoch_8",
    "epoch_9",
    "epoch_10",
]

models = { 'llama3-i':'Llama 3.1 Instruct 8 Billion',
    'llama3-i-1b':'Llama 3.1 Instruct 1 Billion',
    "olmo-i": "OLMo Instruct 7 Billion"
}

In [ ]:
csv_filename = "[tacl] experiment_results.csv"
learnt_entities = {}
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(
        [
            "Model",
            "Strategy",
            "Epoch",
            "#Training Docs",
            "Accum. Time",
            "ppl",
            "Low-Freq",
            "High-Freq",
            "Avg.",
            "MMLU",
            "Math",
            "Social IQA (Reasoning)",
            "#entity with R1>60% out of 100",
            "#of which learnt in previous epoch",
            "#of which learnt in this epoch",
            "#entities 'forgotten' from previous epoch",
        ]
    )
    for model in models:
        # print(models[model])
        baseline_validation_file = (
            "../output/{}/baseline/validation_probes_results.json".format(model)
        )
        with open(baseline_validation_file, "r", encoding="utf-8") as file:
            baseline_validation = json.load(file)
            baseline_entity_perf = [
                round(baseline_validation["low"]["recall"] * 100, 2),
                round(baseline_validation["high"]["recall"] * 100, 2),
                round(baseline_validation["avg"]["recall"] * 100, 2),
            ]

        baseline_control_file = (
            "../output/{}/baseline/control_probes_results.json".format(model)
        )
        with open(baseline_control_file, "r", encoding="utf-8") as file:
            baseline_control = json.load(file)
            baseline_control_perf = [
                round(baseline_control["MMLU"]["exact_match"] * 100, 2),
                round(baseline_control["math"]["exact_match"] * 100, 2),
                round(baseline_control["SocialIQA"]["exact_match"] * 100, 2),
            ]
        # print(model, *baseline_entity_perf,*baseline_control_perf)
        learnt_entities[models[model]] = {}
        learnt_entities[models[model]]["Baseline"] = [
            entity
            for entity in list(baseline_validation["entities"].keys())[1:]
            if baseline_validation["entities"][entity]["recall"] >= 0.6
        ]
        learnt_ones = learnt_entities[models[model]]["Baseline"]
        writer.writerow(
            [
                models[model],
                "Baseline",
                "Epoch 0",
                "",
                "",
                "",
                *baseline_entity_perf,
                *baseline_control_perf,
                "{0} ({1} High, {2} Low)".format(
                    len(learnt_ones),
                    len(list_intersection(high_freq, learnt_ones)),
                    len(learnt_ones) - len(list_intersection(high_freq, learnt_ones)),
                ),
            ]
        )
        for experiment in experiments:
            learnt_docs = [9171]
            counter = 0
            time = 0
            learnt_entities[models[model]][experiments[experiment]] = {}
            experiment_dr = "../output/{0}/{1}".format(model, experiment)
            if "rag" in experiment:
                json_file_path = os.path.join(
                    experiment_dr, "validation_probes_results.json"
                )

                with open(json_file_path, "r", encoding="utf-8") as file:
                    data = json.load(file)
                    epoch_entity_perf = [
                        round(data["low"]["recall"] * 100, 1),
                        round(data["high"]["recall"] * 100, 1),
                        round(data["avg"]["recall"] * 100, 1),
                    ]

                    learnt_entities[models[model]][experiments[experiment]]["Epoch 0"] = [
                        entity
                        for entity in list(data["entities"].keys())[1:]
                        if data["entities"][entity]["recall"] >= 0.6
                    ]
                    learnt_ones = learnt_entities[models[model]][experiments[experiment]]["Epoch 0"]
                    writer.writerow(
                        [
                            models[model],
                            experiments[experiment],
                            "Epoch 0",
                            "",
                            "",
                            "",
                            *epoch_entity_perf,
                            "",
                            "",
                            "",
                            "{0} ({1} High, {2} Low)".format(
                                len(learnt_ones),
                                len(list_intersection(high_freq, learnt_ones)),
                                len(learnt_ones)
                                - len(list_intersection(high_freq, learnt_ones)),
                            ),
                        ]
                    )
            else:
                for epoch in epochs:
                    train_stats = os.path.join(experiment_dr, "train_stats.json")
                    with open(train_stats, "r", encoding="utf-8") as file:
                        data = json.load(file)
                        ppl = round(data[counter]["Train PPL"], 2)
                        learnt_docs.append(data[counter]["# Remaining Documents"])
                        learnt_doc = learnt_docs[counter]
                        epoch_time = datetime.strptime(data[counter]["Time"], "%H:%M:%S.%f")
                        epoch_minutes = epoch_time.hour * 60 + epoch_time.minute
                        time += epoch_minutes
                        counter += 1
                    json_file_path = os.path.join(
                        experiment_dr, epoch, "control_probes_results.json"
                    )
                    with open(json_file_path, "r", encoding="utf-8") as file:
                        data = json.load(file)
                        epoch_control_perf = [
                            round(data["MMLU"]["exact_match"] * 100, 2),
                            round(data["math"]["exact_match"] * 100, 2),
                            round(data["SocialIQA"]["exact_match"] * 100, 2),
                        ]

                    # if "KL" in experiment: json_file_path = os.path.join(experiment_dr, epoch, "validation_probes_nes_results.json")
                    # elif "ENAF" not in experiment: json_file_path = os.path.join(experiment_dr, epoch, "validation_probes_results.json")
                    # else: json_file_path = os.path.join(experiment_dr, epoch, "validation_probes_nes_results.json")
                    json_file_path = os.path.join(
                        experiment_dr, epoch, "validation_probes_results.json"
                    )

                    with open(json_file_path, "r", encoding="utf-8") as file:
                        data = json.load(file)
                        epoch_entity_perf = [
                            round(data["low"]["recall"] * 100, 1),
                            round(data["high"]["recall"] * 100, 1),
                            round(data["avg"]["recall"] * 100, 1),
                        ]

                        learnt_entities[models[model]][experiments[experiment]][epoch] = [
                            entity
                            for entity in list(data["entities"].keys())[1:]
                            if data["entities"][entity]["recall"] >= 0.6
                        ]
                        learnt_ones = learnt_entities[models[model]][
                            experiments[experiment]
                        ][epoch]
                        writer.writerow(
                            [
                                models[model],
                                experiments[experiment],
                                f"Epoch {epoch.split('_')[1]}",
                                learnt_doc,
                                time,
                                ppl,
                                *epoch_entity_perf,
                                *epoch_control_perf,
                                "{0} ({1} High, {2} Low)".format(
                                    len(learnt_ones),
                                    len(list_intersection(high_freq, learnt_ones)),
                                    len(learnt_ones)
                                    - len(list_intersection(high_freq, learnt_ones)),
                                ),
                            ]
                        )

In [ ]:
results_file = "[tacl] experiment_results.csv"
updated_results_file = "[tacl] additional_results.csv"

entity_epochs = [
    "Epoch 1",
    "Epoch 2",
    "Epoch 3",
    "Epoch 4",
    "Epoch 5",
    "Epoch 6",
    "Epoch 7",
    "Epoch 8",
    "Epoch 9",
    "Epoch 10",
]


with open(results_file, mode="r", newline="") as infile, open(
    updated_results_file, mode="w", newline=""
) as outfile:
    reader = csv.reader(infile)
    writer = csv.writer(outfile)
    for row in reader:  # Model Experiment Epoch
        if row[2] in entity_epochs or "RAG" in row[1]:
            learnt_ones = learnt_entities[row[0]][row[1]]
            ep = int(row[2].split(" ")[1])
            if ep == 1 or "RAG" in row[1]:
                prev_epoch_entities = learnt_entities[row[0]]['Baseline']
            else:
                prev_epoch_entities = learnt_entities[row[0]][row[1]]["epoch_{}".format(ep - 1)]
            mutual_ent = list_intersection(
                learnt_entities[row[0]][row[1]]["epoch_{}".format(ep) if "RAG" not in row[1] else "Epoch 0"],
                prev_epoch_entities,
            )
            forgotten_ent = list_difference(
                prev_epoch_entities,
                learnt_entities[row[0]][row[1]]["epoch_{}".format(ep) if "RAG" not in row[1] else "Epoch 0"],
            )
            new_learnt = list_difference(
                learnt_entities[row[0]][row[1]]["epoch_{}".format(ep) if "RAG" not in row[1] else "Epoch 0"],
                prev_epoch_entities,
            )

            row.append(
                "{0} ({1} High, {2} Low)".format(
                    len(mutual_ent),
                    len(list_intersection(mutual_ent, high_freq)),
                    len(mutual_ent) - len(list_intersection(mutual_ent, high_freq)),
                )
            )
            row.append(
                "{0} ({1} High, {2} Low)".format(
                    len(new_learnt),
                    len(list_intersection(new_learnt, high_freq)),
                    len(new_learnt) - len(list_intersection(new_learnt, high_freq)),
                )
            )
            row.append(
                "{0} ({1} High, {2} Low)".format(
                    len(forgotten_ent),
                    len(list_intersection(forgotten_ent, high_freq)),
                    len(forgotten_ent)
                    - len(list_intersection(forgotten_ent, high_freq)),
                )
            )
        writer.writerow(row)

In [ ]:
import pandas as pd
import json
import plotly.express as px


file_path = "[tacl] additional_results.csv"
df = pd.read_csv(file_path)
data = {}

for index, row in df.iterrows():
    model = row["Model"]
    if model not in data:
        data[model] = {}
    experiment = row["Strategy"]
    epoch = f"{row['Epoch']}"
    if epoch == "Baseline":
        experiment, epoch = "Baseline", "Epoch 0"
    if experiment not in data[model]:
        data[model][experiment] = {}
    data[model][experiment][epoch] = {
        "ppl": row["ppl"],
        "docs": row["#Training Docs"],
        "time": row["Accum. Time"],
        "learnt": row["#entity with R1>60% out of 100"],
        "forgotten": row["#entities 'forgotten' from previous epoch"],
        "entity_learning": [row["Low-Freq"], row["High-Freq"], row["Avg."]],
        "generalization": [row["MMLU"], row["Math"], row["Social IQA (Reasoning)"]],
    }
epochs = [
    "Epoch 1",
    "Epoch 2",
    "Epoch 3",
    "Epoch 4",
    "Epoch 5",
    "Epoch 6",
    "Epoch 7",
    "Epoch 8",
    "Epoch 9",
    "Epoch 10",
]

In [ ]:
for model in models:
    entity_learning_last_values = []
    baseline_value = data[models[model]]["Baseline"]["Epoch 0"]["entity_learning"]
    for method in data[models[model]].keys():
        if method != "Baseline":  # Skip baseline itself
            entity_learning_last_values.append(
                {
                    "Method": method,
                    "Epoch": "Epoch 0",
                    "Avg Entity Learning (%)": baseline_value[2],
                }
            )
    for method, data_entry in data[models[model]].items():
        if "Epoch 1" in data_entry:  # Ensuring it's a method with epochs
            for epoch in epochs:
                entity_learning_last_values.append(
                    {
                        "Method": method,
                        "Epoch": epoch,
                        "Avg Entity Learning (%)": data_entry[epoch]["entity_learning"][
                            2
                        ],  # Third value (index 2)
                    }
                )
    df = pd.DataFrame(entity_learning_last_values)
    fig = px.line(
        df,
        x="Epoch",
        y="Avg Entity Learning (%)",
        color="Method",
        markers=True,
        title="Avg Entity Perf(%) {} (Baseline as Epoch 0)".format(models[model]),
    )
    fig.show()

## Vanilla Fine-Tuning

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

display_model = "OLMo Instruct 7 Billion"
method_name = "Vanilla FineTuning (LoRA)"

x_labels = [
    "Epoch 1",
    "Epoch 2",
    "Epoch 3",
    "Epoch 4",
    "Epoch 5",
    "Epoch 6",
    "Epoch 7",
    "Epoch 8",
    "Epoch 9",
    "Epoch 10",
]
entry = data[display_model].get(method_name, {})

baseline = data[display_model]["Baseline"].get("Epoch 0", {})

ppl_vals = [baseline.get("ppl", np.nan)] + [
    entry.get(ep, {}).get("ppl", np.nan) for ep in x_labels
]
avg_entity_vals = [
    round(baseline.get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
] + [
    round(entry.get(ep, {}).get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
    for ep in x_labels
]
avg_generalization_vals = [
    (
        baseline.get("generalization", [np.nan, np.nan, np.nan])[0]
        + baseline.get("generalization", [np.nan, np.nan, np.nan])[2]
    )
    // 2
] + [
    (
        entry.get(ep, {}).get("generalization", [0, None, 0])[0]
        + entry.get(ep, {}).get("generalization", [0, None, 0])[2]
    )
    // 2
    for ep in x_labels
]


plt.style.use("seaborn-v0_8-whitegrid")
colors = {"ppl": "#2b8cbe", "entity": "#31a354", "gen": "#e6550d"}
fig, axes = plt.subplots(
    3, 1, figsize=(8, 9), sharex=True, gridspec_kw={"hspace": 0.12}
)
fig.suptitle(
    f"Training Dynamics of OLMo\n Across Epochs (LoRA)",
    fontsize=21,
    fontweight="bold",
    y=0.96,
)

plots = [
    (axes[0], ppl_vals, "PPL", colors["ppl"]),
    (axes[1], avg_entity_vals, "Knowledge\nAcquisition (%)", colors["entity"]),
    (axes[2], avg_generalization_vals, "Out Of Domain (%)", colors["gen"]),
]

x_labels = ["Epoch 0"] + x_labels
x = np.arange(0, len(x_labels))
for ax, vals, label, col in plots:
    vals_clean = [
        v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))
    ]
    ax.plot(
        x,
        vals,
        marker="o",
        markersize=10,
        linewidth=3,
        color=col,
        markeredgecolor="k",
        markeredgewidth=1,
    )
    ax.set_ylabel(label, fontsize=19, fontweight="semibold", labelpad=15)
    ax.tick_params(axis="y", labelsize=17)
    # set sensible y-limits with padding
    if vals_clean:
        vmin, vmax = min(vals_clean), max(vals_clean)
        pad = max((vmax - vmin) * 0.08, 0.2)
        ax.set_ylim(vmin - pad, vmax + pad)
    ax.grid(alpha=0.7, which="both", linestyle="--")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# X axis formatting
axes[2].set_xticks(x)
axes[2].set_xticklabels(x, fontsize=17)

axes[2].set_xlabel("Epoch (#)", fontsize=19, fontweight="semibold", labelpad=15)

# set y-axis formatters
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter(f"%.1f"))
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter(f"%.0f"))
axes[2].yaxis.set_major_formatter(mticker.FormatStrFormatter(f"%.0f"))

plt.savefig("olmo_training_dynamics.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Methods Comparison

In [ ]:
import numpy as np
from matplotlib.axes import Axes

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

#display_model = 'Llama 3.1 Instruct 1 Billion'
display_model = 'OLMo Instruct 7 Billion'
method_names = [
    "Vanilla FineTuning (LoRA)",
    '+KL MJ|Mj-1',
    '+KL Mj|M0',
    "+Structured Annotation",
    '+Adaptive Exposure',
]

x_labels = [
    "Epoch 1",
    "Epoch 2",
    "Epoch 3",
    "Epoch 4",
    "Epoch 5",
    "Epoch 6",
    "Epoch 7",
    "Epoch 8",
    "Epoch 9",
    "Epoch 10",
]


plt.style.use("seaborn-v0_8-whitegrid")

method_colors = {
    "Vanilla FineTuning (LoRA)": "#d95f02",
    "+Structured Annotation": "#1b9e77",
    "+Adaptive Exposure": "#66a61e",
    "+KL Mj|M0": "#e7298a",
    "+KL MJ|Mj-1": "#7570b3",
}
method_markers = {
    "Vanilla FineTuning (LoRA)": "o",
    "+Structured Annotation": "s",
    "+Adaptive Exposure": "D",
    "+KL Mj|M0": "^",
    "+KL MJ|Mj-1": "v",
}
method_linestyles = {
    "Vanilla FineTuning (LoRA)": "-",
    "+Structured Annotation": "--",
    "+Adaptive Exposure": "-.",
    "+KL Mj|M0": ":",
    "+KL MJ|Mj-1": (0, (3, 1, 1, 1)),
}
method_labeles = {
    "Vanilla FineTuning (LoRA)": "LoRA",
    "+Structured Annotation": "Structured Annotation",
    "+Adaptive Exposure": "Curriculum Learning",
    "+KL Mj|M0": "Pre-training Regularization",
    "+KL MJ|Mj-1": "Step-wise Regularization",
}

fig, axes = plt.subplots(
    1, 3, figsize=(20, 5), sharex=True, gridspec_kw={"wspace": 0.15}
)

#fig.suptitle(
#    f"Training Dynamics of OLMo Across Epochs",
#    fontsize=21,
#    fontweight="bold",
#    y=0.94,
#)

x = np.arange(0, len(x_labels) + 1)

baseline = data[display_model]["Baseline"].get("Epoch 0", {})

for method_name in method_names:
    entry = data[display_model].get(method_name, {})

    ppl_vals = [baseline.get("ppl", np.nan)] + [
        entry.get(ep, {}).get("ppl", np.nan) for ep in x_labels
    ]
    avg_entity_vals = [
        round(baseline.get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
    ] + [
        round(entry.get(ep, {}).get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
        for ep in x_labels
    ]
    avg_generalization_vals = [
        (
            baseline.get("generalization", [np.nan, np.nan, np.nan])[0]
            + baseline.get("generalization", [np.nan, np.nan, np.nan])[2]
        )
        // 2
    ] + [
        (
            entry.get(ep, {}).get("generalization", [0, None, 0])[0]
            + entry.get(ep, {}).get("generalization", [0, None, 0])[2]
        )
        // 2
        for ep in x_labels
    ]


    plots = [
        (axes[0], ppl_vals),
        (axes[1], avg_entity_vals),
        (axes[2], avg_generalization_vals),
    ]

    for ax, vals in plots:
        vals_clean = [
            v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))
        ]
        ax.plot(
            x,
            vals,
            marker=method_markers[method_name],
            markersize=8,
            linewidth=2,
            linestyle=method_linestyles[method_name],
            color=method_colors[method_name],
            markeredgecolor="k",
            markeredgewidth=1,
            label=method_labeles[method_name],
        )

        ax.tick_params(axis="y", labelsize=17)
        ## set sensible y-limits with padding
        ax.grid(alpha=0.7, which="both", linestyle="--")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

# X axis formatting
axes[0].set_xticks(x)
axes[0].set_xticklabels(x, fontsize=19)
axes[0].set_title("PPL $(\\downarrow)$", fontsize=22, fontweight="semibold")
axes[1].set_xticklabels(x, fontsize=19)
axes[1].set_title("Factual Knowledge Recall $(\\uparrow)$", fontsize=22, fontweight="semibold")
axes[2].set_xticks(x)
axes[2].set_xticklabels(x, fontsize=19)
axes[2].set_title("OOD Tasks Accuracy $(\\uparrow)$", fontsize=22, fontweight="semibold")

axes[1].set_xlabel("Epoch (#)", fontsize=22, fontweight="semibold", labelpad=15)

# set y-axis formatters
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter(f"%.1f"))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))

leg = axes[1].legend(
    fontsize=19,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.4),
    ncol=5,
    frameon=True,
    columnspacing=1.0,
    handletextpad=0.6,
)
for txt in leg.get_texts():
    txt.set_ha("center")
    txt.set_multialignment("center")

#plt.savefig("llama3-8b_training_dynamics_different_methods.pdf", dpi=300, bbox_inches="tight")
plt.savefig("olmo_training_dynamics_different_methods.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Methods LLaMA models

In [ ]:
import numpy as np
from matplotlib.axes import Axes

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

method_names = [
    "Vanilla FineTuning (LoRA)",
    '+KL MJ|Mj-1',
    '+KL Mj|M0',
    "+Structured Annotation",
    '+Adaptive Exposure',
]

x_labels = [
    "Epoch 1",
    "Epoch 2",
    "Epoch 3",
    "Epoch 4",
    "Epoch 5",
    "Epoch 6",
    "Epoch 7",
    "Epoch 8",
    "Epoch 9",
    "Epoch 10",
]


plt.style.use("seaborn-v0_8-whitegrid")

method_colors = {
    "Vanilla FineTuning (LoRA)": "#d95f02",
    "+Structured Annotation": "#1b9e77",
    "+Adaptive Exposure": "#66a61e",
    "+KL Mj|M0": "#e7298a",
    "+KL MJ|Mj-1": "#7570b3",
}
method_markers = {
    "Vanilla FineTuning (LoRA)": "o",
    "+Structured Annotation": "s",
    "+Adaptive Exposure": "D",
    "+KL Mj|M0": "^",
    "+KL MJ|Mj-1": "v",
}
method_linestyles = {
    "Vanilla FineTuning (LoRA)": "-",
    "+Structured Annotation": "--",
    "+Adaptive Exposure": "-.",
    "+KL Mj|M0": ":",
    "+KL MJ|Mj-1": (0, (3, 1, 1, 1)),
}
method_labeles = {
    "Vanilla FineTuning (LoRA)": "LoRA",
    "+Structured Annotation": "Structured\nAnnotation",
    "+Adaptive Exposure": "Curriculum\nLearning",
    "+KL Mj|M0": "Pre-training\nRegularization",
    "+KL MJ|Mj-1": "Step-wise\nRegularization",
}



fig, axes = plt.subplots(
    2, 2, figsize=(15, 8), 
    #gridspec_kw={"wspace": 0.80}
    gridspec_kw={"hspace": 0.90, "wspace": 0.15}
)

x = np.arange(0, len(x_labels) + 1)

for i, display_model in enumerate(['Llama 3.1 Instruct 8 Billion', 'Llama 3.1 Instruct 1 Billion']):
    baseline = data[display_model]["Baseline"].get("Epoch 0", {})
    for method_name in method_names:
        entry = data[display_model].get(method_name, {})

        avg_entity_vals = [
            round(baseline.get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
        ] + [
            round(entry.get(ep, {}).get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
            for ep in x_labels
        ]
        avg_generalization_vals = [
            (
                baseline.get("generalization", [np.nan, np.nan, np.nan])[0]
                + baseline.get("generalization", [np.nan, np.nan, np.nan])[2]
            )
            // 2
        ] + [
            (
                entry.get(ep, {}).get("generalization", [0, None, 0])[0]
                + entry.get(ep, {}).get("generalization", [0, None, 0])[2]
            )
            // 2
            for ep in x_labels
        ]


        plots = [
            (axes[i][0], avg_entity_vals),
            (axes[i][1], avg_generalization_vals),
        ]

        for ax, vals in plots:
            vals_clean = [
                v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))
            ]
            ax.plot(
                x,
                vals,
                marker=method_markers[method_name],
                markersize=8,
                linewidth=2,
                linestyle=method_linestyles[method_name],
                color=method_colors[method_name],
                markeredgecolor="k",
                markeredgewidth=1,
                label=method_labeles[method_name],
            )

            ax.tick_params(axis="y", labelsize=21)
            ## set sensible y-limits with padding
            ax.grid(alpha=0.7, which="both", linestyle="--")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

# X axis formatting
#axes[0][0].set_title("Knowledge Acquisition (%)", fontsize=22, fontweight="semibold")
#axes[0][1].set_title("Out Of Domain (%)", fontsize=22, fontweight="semibold")
axes[0][0].set_title("Factual Knowledge Recall $(\\uparrow)$", fontsize=27, fontweight="semibold", pad=15)
axes[0][1].set_title("OOD Tasks Accuracy $(\\uparrow)$", fontsize=27, fontweight="semibold", pad=15)
axes[1][0].set_title("Factual Knowledge Recall $(\\uparrow)$", fontsize=27, fontweight="semibold", pad=15)
axes[1][1].set_title("OOD Tasks Accuracy $(\\uparrow)$", fontsize=27, fontweight="semibold", pad=15)

axes[0][0].set_xticks(x)
axes[0][1].set_xticks(x)
axes[1][0].set_xticks(x)
axes[1][1].set_xticks(x)
axes[0][0].set_xticklabels(x, fontsize=21)
axes[0][1].set_xticklabels(x, fontsize=21)
axes[1][0].set_xticklabels(x, fontsize=21)
axes[1][1].set_xticklabels(x, fontsize=21)

#axes[1][0].set_xlabel("Epoch (#)", fontsize=27, fontweight="semibold", labelpad=15)
axes[1][1].set_xlabel("Epoch (#)", fontsize=27, fontweight="semibold")
# move xlabel further left (x in axes coords: 0=left, 1=right)
axes[1][1].xaxis.set_label_coords(-0.08, -0.25)

axes[0][0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))
axes[0][1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))
axes[1][0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))
axes[1][1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))

leg = axes[1][1].legend(
    fontsize=20,
    #loc="center",
    #bbox_to_anchor=(-0.45, 1),
    #ncol=1,
    loc="center",
    bbox_to_anchor=(-0.10, 1.5),
    ncol=5,
    frameon=True,
    columnspacing=1.0,
    handletextpad=0.6,
)
for txt in leg.get_texts():
    txt.set_ha("center")
    txt.set_multialignment("center")


plt.savefig("llama-3_training_dynamics_different_methods.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
from matplotlib.axes import Axes

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

method_names = [
    "Vanilla FineTuning (LoRA)",
    "+KL MJ|Mj-1",
    "+KL Mj|M0",
    "+Structured Annotation",
    "+Adaptive Exposure",
]

x_labels = [
    "Epoch 1",
    "Epoch 2",
    "Epoch 3",
    "Epoch 4",
    "Epoch 5",
    "Epoch 6",
    "Epoch 7",
    "Epoch 8",
    "Epoch 9",
    "Epoch 10",
]


plt.style.use("seaborn-v0_8-whitegrid")

method_colors = {
    "Vanilla FineTuning (LoRA)": "#d95f02",
    "+Structured Annotation": "#1b9e77",
    "+Adaptive Exposure": "#66a61e",
    "+KL Mj|M0": "#e7298a",
    "+KL MJ|Mj-1": "#7570b3",
}
method_markers = {
    "Vanilla FineTuning (LoRA)": "o",
    "+Structured Annotation": "s",
    "+Adaptive Exposure": "D",
    "+KL Mj|M0": "^",
    "+KL MJ|Mj-1": "v",
}
method_linestyles = {
    "Vanilla FineTuning (LoRA)": "-",
    "+Structured Annotation": "--",
    "+Adaptive Exposure": "-.",
    "+KL Mj|M0": ":",
    "+KL MJ|Mj-1": (0, (3, 1, 1, 1)),
}
method_labeles = {
    "Vanilla FineTuning (LoRA)": "LoRA",
    "+Structured Annotation": "Structured\nAnnotation",
    "+Adaptive Exposure": "Curriculum\nLearning",
    "+KL Mj|M0": "Pre-training\nRegularization",
    "+KL MJ|Mj-1": "Step-wise\nRegularization",
}


fig, axes = plt.subplots(2, figsize=(9, 12))
fig.subplots_adjust(
    hspace=0.80
)

x = np.arange(0, len(x_labels) + 1)

display_model = "Llama 3.1 Instruct 8 Billion"  # , 'Llama 3.1 Instruct 1 Billion'
baseline = data[display_model]["Baseline"].get("Epoch 0", {})
for method_name in method_names:
    entry = data[display_model].get(method_name, {})

    avg_entity_vals = [
        round(baseline.get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
    ] + [
        round(entry.get(ep, {}).get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
        for ep in x_labels
    ]
    avg_generalization_vals = [
        (
            baseline.get("generalization", [np.nan, np.nan, np.nan])[0]
            + baseline.get("generalization", [np.nan, np.nan, np.nan])[2]
        )
        // 2
    ] + [
        (
            entry.get(ep, {}).get("generalization", [0, None, 0])[0]
            + entry.get(ep, {}).get("generalization", [0, None, 0])[2]
        )
        // 2
        for ep in x_labels
    ]

    plots = [
        (axes[0], avg_entity_vals),
        (axes[1], avg_generalization_vals),
    ]

    for ax, vals in plots:
        vals_clean = [
            v
            for v in vals
            if v is not None and not (isinstance(v, float) and np.isnan(v))
        ]
        ax.plot(
            x,
            vals,
            marker=method_markers[method_name],
            markersize=8,
            linewidth=2,
            linestyle=method_linestyles[method_name],
            color=method_colors[method_name],
            markeredgecolor="k",
            markeredgewidth=1,
            label=method_labeles[method_name],
        )

        ax.tick_params(axis="y", labelsize=21)
        ## set sensible y-limits with padding
        ax.grid(alpha=0.7, which="both", linestyle="--")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

# X axis formatting
axes[0].set_title(
    "Factual Knowledge Recall $(\\uparrow)$", fontsize=27, fontweight="semibold", pad=15
)
axes[1].set_title("OOD Tasks Accuracy $(\\uparrow)$", fontsize=27, fontweight="semibold", pad=15)

axes[0].set_xticks(x)
axes[0].set_xticklabels(x, fontsize=21)
axes[1].set_xticks(x)
axes[1].set_xticklabels(x, fontsize=21)

axes[1].set_xlabel("Epoch (#)", fontsize=27, fontweight="semibold", labelpad=15)

# show percentage sign on y ticks
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))

ncols = 3
nlines = 5

f = lambda x, p: np.sin(x * p)

kw = dict(
    framealpha=1,
    bbox_to_anchor=(0.46, 1.16),
    fontsize=20,
    frameon=True,
    columnspacing=1.5,
    handletextpad=0.6,
)
leg1 = axes[1].legend(
    handles=axes[1].lines[: nlines // ncols * ncols],
    ncol=ncols,
    loc="lower center",
    **kw,
)
for txt in leg1.get_texts():
    txt.set_ha("center")
    txt.set_multialignment("center")
plt.gca().add_artist(leg1)

leg2 = axes[1].legend(
    handles=axes[1].lines[nlines // ncols * ncols :],
    ncol=nlines - nlines // ncols * ncols,
    **kw,
)
for txt in leg2.get_texts():
    txt.set_ha("center")
    txt.set_multialignment("center")

leg2.remove()
leg1._legend_box._children.append(leg2._legend_handle_box)
leg1._legend_box.stale = True


plt.savefig(
    "llama-3-8b_training_dynamics_knowledge_and_ood.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

## Entity Heatmap

In [ ]:
experiments = {
    "VFT": "Vanilla FineTuning (LoRA)",
    "VFT_ENAF": "+Structured Annotation",
    "EFT": "+Adaptive Exposure",
    "VFT_KL_10": "+KL Mj|M0",
    "VFT_KL_LAST_1": "+KL MJ|Mj-1",
}
epochs = [
    "epoch_1",
    "epoch_2",
    "epoch_3",
    "epoch_4",
    "epoch_5",
    "epoch_6",
    "epoch_7",
    "epoch_8",
    "epoch_9",
    "epoch_10",
]
models = {  'llama3-i':'Llama 3.1 Instruct 8 Billion',
    'llama3-i-1b':'Llama 3.1 Instruct 1 Billion',
    "olmo-i": "OLMo Instruct 7 Billion"
}

entities_performance = {}
for model in models:
    # print(models[model])
    baseline_validation_file = (
        "../output/{}/baseline/validation_probes_results.json".format(model)
    )
    entities_performance[models[model]] = {}
    entities_performance[models[model]]['Baseline'] = {} 
    entities_performance[models[model]]['Baseline']['Epoch 0'] = {}
    for entity, ent_stats in baseline_validation["entities"].items():
        if not isinstance(ent_stats, dict):
            continue
        entities_performance[models[model]]['Baseline']['Epoch 0'][entity] = ent_stats["recall"]

    for experiment in experiments:
        entities_performance[models[model]][experiments[experiment]] = {}
        experiment_dr = "../output/{0}/{1}".format(model, experiment)
        for epoch in epochs:
            entities_performance[models[model]][experiments[experiment]][epoch] = {}
            json_file_path = os.path.join(
                experiment_dr, epoch, "validation_probes_results.json"
            )

            with open(json_file_path, "r", encoding="utf-8") as file:
                data = json.load(file)

                for entity, ent_stats in data["entities"].items():
                    if not isinstance(ent_stats, dict):
                        continue
                    entities_performance[models[model]][experiments[experiment]][epoch][entity] = data["entities"][entity]["recall"]
                learnt_ones = entities_performance[models[model]][
                    experiments[experiment]
                ][epoch]

In [ ]:
from matplotlib import colors as mcolors
import numpy as np

#model = "Llama 3.1 Instruct 1 Billion"
model = "OLMo Instruct 7 Billion"
method = "Vanilla FineTuning (LoRA)"
entity = "PERSON"

# split high / low entities and build matrices for each side
x_high = list(entities[entity]["high"].keys())
x_low = list(entities[entity]["low"].keys())

method_epochs = sorted(
    entities_performance[model][method].keys(),
    key=lambda s: int(s.split("_")[1]),
    reverse=True,
)
sorted_epochs = method_epochs + ["0"]


def build_matrix(ent_list):
    rows = []
    for ep in sorted_epochs:
        if ep == "0":
            row = [
                entities_performance[model]["Baseline"]["Epoch 0"].get(ent, np.nan)
                * 100
                for ent in ent_list
            ]
        else:
            row = [
                entities_performance[model][method][ep].get(ent, np.nan) * 100
                for ent in ent_list
            ]
        rows.append(row)
    return np.array(rows, dtype=float)


mat_high = build_matrix(x_high)
mat_low = build_matrix(x_low)

# horizontal subplot: high on the left, low on the right
fig, (axl, axr) = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
plt.style.use("seaborn-v0_8-whitegrid")

cmap_name = "plasma"
cmap = plt.get_cmap(cmap_name)
norm = mcolors.Normalize(vmin=0, vmax=100)

im1 = axl.imshow(mat_high, aspect="auto", cmap=cmap, norm=norm)
im2 = axr.imshow(mat_low, aspect="auto", cmap=cmap, norm=norm)

# annotate both heatmaps
# for ax, mat in [(axl, mat_high), (axr, mat_low)]:
#    for i in range(mat.shape[0]):
#        for j in range(mat.shape[1]):
#            val = mat[i, j]
#            if np.isnan(val):
#                continue
#            txt = f"{val:.0f}"
#            text_color = "white" if val < 50 else "black"
#            ax.text(j, i, txt, ha="center", va="center", color=text_color, fontsize=12)

# ticks and labels
axl.set_xticks(np.arange(len(x_high)))
shortened_labels = {
    "Stephen Thomas Erlewine": "S. Thomas Erlewine",
    "the United States": "USA",
    "New York City": "NYC",
    "Central African Republic": "Cen. African Rep.",



}
axl.set_xticklabels(
    [
        shortened_labels.get(ent, ent)
        for ent in x_high
    ],
    rotation=90,
    fontsize=15,
)
axr.set_xticks(np.arange(len(x_low)))
axr.set_xticklabels(
    [
        shortened_labels.get(ent, ent)
        for ent in x_low
    ], rotation=90, fontsize=15)

yticks = [ep.replace("epoch_", "") for ep in sorted_epochs]
axl.set_yticks(np.arange(len(sorted_epochs)))
axl.set_yticklabels(yticks, fontsize=15)

axl.set_title(f"High frequency {entity}", fontsize=22, fontweight="bold")
axr.set_title(f"Low frequency {entity}", fontsize=22, fontweight="bold")
# axl.set_xlabel("Entities")
# axr.set_xlabel("Entities")
axl.set_ylabel("Epoch", fontsize=19, fontweight="semibold")


axl.grid(False)
axr.grid(False)

plt.tight_layout()
# shared colorbar
cbar = fig.colorbar(
    im2,
    ax=[axl, axr],
    fraction=0.04,
    pad=0.02,
    ticks=[0, 20, 40, 60, 80, 100],
)
cbar.ax.tick_params(labelsize=15)          # colorbar tick labels
#cbar.set_label("Recall (%)", fontsize=18)  # colorbar label
if model == "OLMo Instruct 7 Billion":
    model_name = "olmo"
elif model == "Llama 3.1 Instruct 1 Billion":
    model_name = "llama3-1b"
else:
    model_name = "llama3-8b"
plt.savefig(
    f"{model_name}_{method.replace(' ','_').replace('|','')}_{entity}_heatmap.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Entity Frequency

In [ ]:
import json

with open("ne_counts_sorted.json", "r", encoding="utf-8") as file:
    ne_counts = json.load(file)

In [ ]:
with open("accepted_entities.json", "r", encoding="utf-8") as file:
    accepted_entities = json.load(file)

In [ ]:
from tqdm import tqdm

high_freq_person = {}
low_freq_person = {}
high_freq_gpe = {}
low_freq_gpe = {}

for ne_key, ne_count in tqdm(ne_counts):
    chunks = ne_key[1:-1].split(', ')
    ne_type = chunks[-1][1:-1]
    ne = ", ".join(chunks[:-1])[1:-1]
    if ne_type in accepted_entities:
        if ne in accepted_entities[ne_type]["high"]:
            if ne_type == "PERSON":
                high_freq_person[ne] = ne_count
            elif ne_type == "GPE":
                high_freq_gpe[ne] = ne_count
        elif ne in accepted_entities[ne_type]["low"]:
            if ne_type == "PERSON":
                low_freq_person[ne] = ne_count
            elif ne_type == "GPE":
                low_freq_gpe[ne] = ne_count

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


plt.style.use("seaborn-v0_8-whitegrid")

plt.bar(
    high_freq_person.keys(), # type: ignore
    high_freq_person.values(), # type: ignore
    color="blue",
    label="High Frequency",
)
plt.bar(
    low_freq_person.keys(), # type: ignore
    low_freq_person.values(), # type: ignore
    color="orange",
    label="Low Frequency",
)

plt.title(
    f"Frequency for PERSON Entities",
    fontsize=21,
    fontweight="bold",
    y=1.05,
)

plt.ylabel("Frequency", fontsize=19, fontweight="semibold", labelpad=15)
plt.tick_params(axis="y", labelsize=17)
plt.yscale("log")

#ax = plt.gca()
#ax.yaxis.set_major_formatter(
#    mticker.FuncFormatter(lambda x, pos: f"{int(x/1000)}K" if abs(x) >= 1000 else f"{int(x)}")
#)

plt.xlabel("Entities", fontsize=19, fontweight="semibold", labelpad=15)
plt.xticks([])
#plt.xticks(rotation=45, ha="right", fontsize=12)

plt.legend(frameon=True, facecolor="white", fontsize=12, framealpha=1)
plt.grid(alpha=0.7, which="major", linestyle="--")

plt.tight_layout()
plt.savefig("entity_frequency_person.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


plt.style.use("seaborn-v0_8-whitegrid")

plt.bar(
    high_freq_gpe.keys(), # type: ignore
    high_freq_gpe.values(), # type: ignore
    color="blue",
    label="High Frequency",
)
plt.bar(
    low_freq_gpe.keys(), # type: ignore
    low_freq_gpe.values(), # type: ignore
    color="orange",
    label="Low Frequency",
)

plt.title(
    f"Frequency for GPE Entities",
    fontsize=21,
    fontweight="bold",
    y=1.05,
)

plt.ylabel("Frequency", fontsize=19, fontweight="semibold", labelpad=15)
plt.tick_params(axis="y", labelsize=17)
plt.yscale("log")

#ax = plt.gca()
#ax.yaxis.set_major_formatter(
#    mticker.FuncFormatter(lambda x, pos: f"{int(x/1000)}K" if abs(x) >= 1000 else f"{int(x)}")
#)


plt.xlabel("Entities", fontsize=19, fontweight="semibold", labelpad=15)
plt.xticks([])
#plt.xticks(rotation=45, ha="right", fontsize=12)


plt.legend(frameon=True, facecolor="white", fontsize=12, framealpha=1)
plt.grid(alpha=0.7, which="major", linestyle="--")

plt.tight_layout()
plt.savefig("entity_frequency_gpe.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Combined vertical subplot for PERSON and GPE entity frequency plots

fig, axes = plt.subplots(2, 1, figsize=(6, 6), sharex=False)
plt.style.use("seaborn-v0_8-whitegrid")

# Top: PERSON
ax = axes[0]
ax.bar(
    list(high_freq_person.keys()),  # type: ignore
    list(high_freq_person.values()),  # type: ignore
    color="blue",
    label="High Frequency",
    alpha=0.9,
)
ax.bar(
    list(low_freq_person.keys()),  # type: ignore
    list(low_freq_person.values()),  # type: ignore
    color="orange",
    label="Low Frequency",
    alpha=0.9,
)
#ax.set_title("Frequency for PERSON Entities", fontsize=19, fontweight="bold", y=1.02)
ax.set_xlabel("PERSON Entities", fontsize=19, labelpad=5, fontweight="semibold")
ax.set_ylabel("Frequency", fontsize=19, labelpad=15, fontweight="semibold")
ax.set_yscale("log")
ax.legend(frameon=True, fontsize=12)
ax.grid(alpha=0.6, which="major", linestyle="--")
ax.set_xticks([])
ax.tick_params(axis="y", labelsize=15)

# Bottom: GPE
ax = axes[1]
ax.bar(
    list(high_freq_gpe.keys()),  # type: ignore
    list(high_freq_gpe.values()),  # type: ignore
    color="blue",
    label="High Frequency",
    alpha=0.9,
)
ax.bar(
    list(low_freq_gpe.keys()),  # type: ignore
    list(low_freq_gpe.values()),  # type: ignore
    color="orange",
    label="Low Frequency",
    alpha=0.9,
)
#ax.set_title("Frequency for GPE Entities", fontsize=19, fontweight="bold", y=1.02)
ax.set_xlabel("GPE Entities", fontsize=19, fontweight="semibold", labelpad=5)
ax.set_ylabel("Frequency", fontsize=19, labelpad=15, fontweight="semibold")
ax.set_yscale("log")
ax.legend(frameon=True, fontsize=12)
ax.grid(alpha=0.6, which="major", linestyle="--")
ax.set_xticks([])
ax.tick_params(axis="y", labelsize=15)

plt.tight_layout()
plt.savefig("entity_frequency_combined.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Cheating Experiment

In [ ]:
experiments = {
    "VFT_MAIN": "Vanilla FineTuning (LoRA) on Main Docs",
}
epochs = [f"epoch_{i}" for i in range(1, 101)]

models = {"llama3-i-1b": "Llama 3.1 Instruct 1 Billion"}

In [ ]:
csv_filename = "[tacl] experiment_results.csv"
learnt_entities = {}
with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(
        [
            "Model",
            "Strategy",
            "Epoch",
            "#Training Docs",
            "Accum. Time",
            "ppl",
            "Low-Freq",
            "High-Freq",
            "Avg.",
            "MMLU",
            "Math",
            "Social IQA (Reasoning)",
            "#entity with R1>60% out of 100",
            "#of which learnt in previous epoch",
            "#of which learnt in this epoch",
            "#entities 'forgotten' from previous epoch",
        ]
    )
    for model in models:
        baseline_validation_file = (
            "../output/{}/baseline/validation_probes_results.json".format(model)
        )
        with open(baseline_validation_file, "r", encoding="utf-8") as file:
            baseline_validation = json.load(file)
            baseline_entity_perf = [
                round(baseline_validation["low"]["recall"] * 100, 2),
                round(baseline_validation["high"]["recall"] * 100, 2),
                round(baseline_validation["avg"]["recall"] * 100, 2),
            ]

        baseline_control_file = (
            "../output/{}/baseline/control_probes_results.json".format(model)
        )
        with open(baseline_control_file, "r", encoding="utf-8") as file:
            baseline_control = json.load(file)
            baseline_control_perf = [
                round(baseline_control["MMLU"]["exact_match"] * 100, 2),
                round(baseline_control["math"]["exact_match"] * 100, 2),
                round(baseline_control["SocialIQA"]["exact_match"] * 100, 2),
            ]
        learnt_entities[models[model]] = {}
        learnt_entities[models[model]]["Baseline"] = [
            entity
            for entity in list(baseline_validation["entities"].keys())[1:]
            if baseline_validation["entities"][entity]["recall"] >= 0.6
        ]
        learnt_ones = learnt_entities[models[model]]["Baseline"]
        writer.writerow(
            [
                models[model],
                "Baseline",
                "Epoch 0",
                "",
                "",
                "",
                *baseline_entity_perf,
                *baseline_control_perf,
                "{0} ({1} High, {2} Low)".format(
                    len(learnt_ones),
                    len(list_intersection(high_freq, learnt_ones)),
                    len(learnt_ones) - len(list_intersection(high_freq, learnt_ones)),
                ),
            ]
        )
        for experiment in experiments:
            learnt_docs = [9171]
            counter = 0
            time = 0
            learnt_entities[models[model]][experiments[experiment]] = {}
            experiment_dr = "../output/{0}/{1}".format(model, experiment)
            for epoch in epochs:
                train_stats = os.path.join(experiment_dr, "train_stats.json")
                with open(train_stats, "r", encoding="utf-8") as file:
                    data = json.load(file)
                    ppl = round(data[counter]["Train PPL"], 2)
                    learnt_docs.append(data[counter]["# Remaining Documents"])
                    learnt_doc = learnt_docs[counter]
                    try:
                        epoch_time = datetime.strptime(data[counter]["Time"], "%H:%M:%S.%f")
                        epoch_minutes = epoch_time.hour * 60 + epoch_time.minute
                    except:
                        epoch_minutes = 0
                    time += epoch_minutes
                    counter += 1
                json_file_path = os.path.join(
                    experiment_dr, epoch, "control_probes_results.json"
                )
                if not os.path.exists(json_file_path):
                    epoch_control_perf = [
                        None, None, None
                    ]
                else:
                    with open(json_file_path, "r", encoding="utf-8") as file:
                        data = json.load(file)
                        epoch_control_perf = [
                            round(data["MMLU"]["exact_match"] * 100, 2),
                            round(data["math"]["exact_match"] * 100, 2),
                            round(data["SocialIQA"]["exact_match"] * 100, 2),
                        ]

                # if "KL" in experiment: json_file_path = os.path.join(experiment_dr, epoch, "validation_probes_nes_results.json")
                # elif "ENAF" not in experiment: json_file_path = os.path.join(experiment_dr, epoch, "validation_probes_results.json")
                # else: json_file_path = os.path.join(experiment_dr, epoch, "validation_probes_nes_results.json")
                json_file_path = os.path.join(
                    experiment_dr, epoch, "validation_probes_results.json"
                )

                with open(json_file_path, "r", encoding="utf-8") as file:
                    data = json.load(file)
                    epoch_entity_perf = [
                        round(data["low"]["recall"] * 100, 1),
                        round(data["high"]["recall"] * 100, 1),
                        round(data["avg"]["recall"] * 100, 1),
                    ]

                    learnt_entities[models[model]][experiments[experiment]][epoch] = [
                        entity
                        for entity in list(data["entities"].keys())[1:]
                        if data["entities"][entity]["recall"] >= 0.6
                    ]
                    learnt_ones = learnt_entities[models[model]][
                        experiments[experiment]
                    ][epoch]
                    writer.writerow(
                        [
                            models[model],
                            experiments[experiment],
                            f"Epoch {epoch.split('_')[1]}",
                            learnt_doc,
                            time,
                            ppl,
                            *epoch_entity_perf,
                            *epoch_control_perf,
                            "{0} ({1} High, {2} Low)".format(
                                len(learnt_ones),
                                len(list_intersection(high_freq, learnt_ones)),
                                len(learnt_ones)
                                - len(list_intersection(high_freq, learnt_ones)),
                            ),
                        ]
                    )

In [ ]:
results_file = "[tacl] experiment_results.csv"
updated_results_file = "[tacl] additional_results.csv"

entity_epochs = [
    f"Epoch {i}" for i in range(1, 101)
]


with open(results_file, mode="r", newline="") as infile, open(
    updated_results_file, mode="w", newline=""
) as outfile:
    reader = csv.reader(infile)
    writer = csv.writer(outfile)
    for row in reader:  # Model Experiment Epoch
        if row[2] in entity_epochs or "RAG" in row[1]:
            learnt_ones = learnt_entities[row[0]][row[1]]
            ep = int(row[2].split(" ")[1])
            if ep == 1 or "RAG" in row[1]:
                prev_epoch_entities = learnt_entities[row[0]]['Baseline']
            else:
                prev_epoch_entities = learnt_entities[row[0]][row[1]]["epoch_{}".format(ep - 1)]
            mutual_ent = list_intersection(
                learnt_entities[row[0]][row[1]]["epoch_{}".format(ep) if "RAG" not in row[1] else "Epoch 0"],
                prev_epoch_entities,
            )
            forgotten_ent = list_difference(
                prev_epoch_entities,
                learnt_entities[row[0]][row[1]]["epoch_{}".format(ep) if "RAG" not in row[1] else "Epoch 0"],
            )
            new_learnt = list_difference(
                learnt_entities[row[0]][row[1]]["epoch_{}".format(ep) if "RAG" not in row[1] else "Epoch 0"],
                prev_epoch_entities,
            )

            row.append(
                "{0} ({1} High, {2} Low)".format(
                    len(mutual_ent),
                    len(list_intersection(mutual_ent, high_freq)),
                    len(mutual_ent) - len(list_intersection(mutual_ent, high_freq)),
                )
            )
            row.append(
                "{0} ({1} High, {2} Low)".format(
                    len(new_learnt),
                    len(list_intersection(new_learnt, high_freq)),
                    len(new_learnt) - len(list_intersection(new_learnt, high_freq)),
                )
            )
            row.append(
                "{0} ({1} High, {2} Low)".format(
                    len(forgotten_ent),
                    len(list_intersection(forgotten_ent, high_freq)),
                    len(forgotten_ent)
                    - len(list_intersection(forgotten_ent, high_freq)),
                )
            )
        writer.writerow(row)

In [ ]:
import pandas as pd
import json
import plotly.express as px


file_path = "[tacl] additional_results.csv"
df = pd.read_csv(file_path)
data = {}

for index, row in df.iterrows():
    model = row["Model"]
    if model not in data:
        data[model] = {}
    experiment = row["Strategy"]
    epoch = f"{row['Epoch']}"
    if epoch == "Baseline":
        experiment, epoch = "Baseline", "Epoch 0"
    if experiment not in data[model]:
        data[model][experiment] = {}
    data[model][experiment][epoch] = {
        "ppl": row["ppl"],
        "docs": row["#Training Docs"],
        "time": row["Accum. Time"],
        "learnt": row["#entity with R1>60% out of 100"],
        "forgotten": row["#entities 'forgotten' from previous epoch"],
        "entity_learning": [row["Low-Freq"], row["High-Freq"], row["Avg."]],
        "generalization": [row["MMLU"], row["Math"], row["Social IQA (Reasoning)"]],
    }
epochs = [
    f"Epoch {i}" for i in range(1, 101)
]

In [ ]:
import numpy as np
from matplotlib.axes import Axes

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

display_model = 'Llama 3.1 Instruct 1 Billion'
method_names = [
    "Vanilla FineTuning (LoRA) on Main Docs",
]

x_labels = [
    f"Epoch {i}" for i in range(1, 101)
]


plt.style.use("seaborn-v0_8-whitegrid")

method_colors = {
    "Vanilla FineTuning (LoRA) on Main Docs": "#d95f02",
}
method_markers = {
    "Vanilla FineTuning (LoRA) on Main Docs": "o",
}
method_linestyles = {
    "Vanilla FineTuning (LoRA) on Main Docs": "-",
}
method_labeles = {
    "Vanilla FineTuning (LoRA) on Main Docs": "Vanilla FineTuning\n(LoRA)",
}

fig, axes = plt.subplots(
    1, 3, figsize=(20, 5), sharex=True, gridspec_kw={"wspace": 0.15}
)

#fig.suptitle(
#    f"Training Dynamics of OLMo Across Epochs",
#    fontsize=21,
#    fontweight="bold",
#    y=0.94,
#)

x = np.arange(0, len(x_labels) + 1)

baseline = data[display_model]["Baseline"].get("Epoch 0", {})

for method_name in method_names:
    entry = data[display_model].get(method_name, {})

    ppl_vals = [baseline.get("ppl", np.nan)] + [
        entry.get(ep, {}).get("ppl", np.nan) for ep in x_labels
    ]
    avg_entity_vals = [
        round(baseline.get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
    ] + [
        round(entry.get(ep, {}).get("entity_learning", [np.nan, np.nan, np.nan])[2], 0)
        for ep in x_labels
    ]
    avg_generalization_vals = [
        (
            baseline.get("generalization", [np.nan, np.nan, np.nan])[0]
            + baseline.get("generalization", [np.nan, np.nan, np.nan])[2]
        )
        // 2
    ] + [
        (
            entry.get(ep, {}).get("generalization", [0, None, 0])[0]
            + entry.get(ep, {}).get("generalization", [0, None, 0])[2]
        )
        // 2
        for ep in x_labels
    ]


    plots = [
        (axes[0], ppl_vals),
        (axes[1], avg_entity_vals),
        (axes[2], avg_generalization_vals),
    ]

    for ax, vals in plots:
        vals_clean = []
        idxs = []
        idx = 0
        for v in vals:
            if v is not None and not (isinstance(v, float) and np.isnan(v)):
                vals_clean.append(v)
                idxs.append(idx)
            idx += 1
        ax.plot(
            idxs,
            vals_clean,
            #marker=method_markers[method_name],
            #markersize=3,
            linewidth=2,
            linestyle=method_linestyles[method_name],
            color=method_colors[method_name],
            markeredgecolor="k",
            markeredgewidth=1,
            label=method_labeles[method_name],
        )

        ax.tick_params(axis="y", labelsize=17)
        ## set sensible y-limits with padding
        ax.grid(alpha=0.7, which="both", linestyle="--")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

# X axis formatting
to_show = list(range(0, 101, 10))
axes[0].set_xticks(to_show)
axes[0].set_xticklabels(to_show, fontsize=19)
axes[0].set_title("PPL $(\\downarrow)$", fontsize=22, fontweight="semibold")
axes[1].set_xticklabels(to_show, fontsize=19)
axes[1].set_title("Factual Knowledge Recall $(\\uparrow)$", fontsize=22, fontweight="semibold")
axes[2].set_xticks(to_show)
axes[2].set_xticklabels(to_show, fontsize=19)
axes[2].set_title("OOD Tasks Accuracy $(\\uparrow)$", fontsize=22, fontweight="semibold")

axes[1].set_xlabel("Epoch (#)", fontsize=22, fontweight="semibold", labelpad=15)

# set y-axis formatters
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter(f"%.1f"))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v:.0f}%"))

#leg = axes[1].legend(
#    fontsize=19,
#    loc="upper right",
#    frameon=True,
#    columnspacing=1.0,
#    handletextpad=0.6,
#)
#for txt in leg.get_texts():
#    txt.set_ha("center")
#    txt.set_multialignment("center")

plt.savefig("llama3-1b_training_dynamics_100_epochs.pdf", dpi=300, bbox_inches="tight")
plt.show()